# Book Recommender System

This notebook documents the development of the rating-based recommendation engine to support the user story: "As a user, I want to see a personalised "For You" feed of book cards so that I can discover books tailored to my preferences." 

The recommender uses a pipleine which implements two strategies:
1. If the user has fewer than 5 book ratings, popularity-based recommendations are produced using a Bayesian average score.
2. Otherwise collaborative filtering via Singular Value Decomposition (SVD) is used

Both strategies are served through `recommender_api.py` which is a Flask API that integrates with the Spring Boot backend, updating recommendations automatically as users rate books. The more the users rate, the more personalised the recommendations become.

## Imports

In [5]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sqlalchemy import create_engine
from surprise import SVD, Reader, Dataset
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate
from sqlalchemy import text

## Loading Goodreads UCSD Book Graph Datasets

This dataset was chosen based on the intended target audience of this application - young adults who follow social media book communities (e.g. BookTok).

In [6]:
#dataset has 93,398 books
books_ya = pd.read_json("../datasets/goodreads_books_young_adult.json.gz", lines=True)

#dataset has 34,919,254 interactions - interactions read will be truncated to simplify model
interactions = pd.read_json("../datasets/goodreads_interactions_young_adult.json.gz", lines=True, nrows=2000000)

books_ya.to_csv("../datasets/goodreads_books_young_adult.csv", index = False)
interactions.to_csv("../datasets/interactions_young_adult.csv", index = False)

In [7]:
#loading author metadata
authors_df = pd.read_json( "../datasets/goodreads_book_authors.json.gz", lines=True)
authors_df.to_csv("../datasets/goodreads_book_authors.csv", index = False)

### Data preprocessing

The raw data requires several cleaning steps before it can be used for modelling...

In [8]:
raw_df = interactions.merge(books_ya, on="book_id")

In [9]:
#removing unnecessary columns

merged_df = raw_df[[
    "user_id",
    "book_id",
    "rating",
    "title",
    "authors",
    "description",
    "publication_year",
    "num_pages",
    "average_rating",
    "ratings_count",
    "image_url",
    "popular_shelves", # to extract the genre
    "language_code",
]].copy()

In [10]:
print(f"Merged dataset shape: {merged_df.shape}")

Merged dataset shape: (2000000, 13)


### Merging author data with dataset

In [11]:
#loading author data 
authors_df = pd.read_json("../datasets/goodreads_book_authors.json.gz", lines = True)
authors_df.to_csv("../datasets/goodreads_book_authors.csv", index = False)

In [12]:
#converting both to string
authors_df["author_id"] = authors_df["author_id"].astype(str)

In [13]:
#extracting author name
merged_df["author_id"] = merged_df["authors"].apply(
  lambda x: x[0]["author_id"] if isinstance(x, list) and len(x) > 0 else None
)

In [14]:
authored_df = merged_df.merge(authors_df[["author_id","name"]], on="author_id", how="left")

In [15]:
authored_df = authored_df.rename(columns={"name":"author"})

In [16]:
#i only need author names 
authored_df = authored_df.drop(columns=["authors", "author_id"])

In [17]:
authored_df[["title","author"]]

,title,author
0,Popular: Vintage Wisdom for a Modern Geek,Maya Van Wagenen
1,Popular: Vintage Wisdom for a Modern Geek,Maya Van Wagenen
2,Popular: Vintage Wisdom for a Modern Geek,Maya Van Wagenen
3,Popular: Vintage Wisdom for a Modern Geek,Maya Van Wagenen
4,Popular: Vintage Wisdom for a Modern Geek,Maya Van Wagenen
...,...,...
1999995,The Devil You Know,Trish Doller
1999996,Irresistible,Liz Bankes
1999997,"The Lost Kingdom (The Elements Series, #1)",Effie Crescent
1999998,Peeps (Peeps #1),Scott Westerfeld


In [18]:
#defining genre shelves to match against
GENRE_SHELVES = {
    "fantasy", "romance", "science-fiction", "sci-fi", "mystery", "thriller",
    "horror", "dystopia", "paranormal", "urban-fantasy", "historical-fiction",
    "contemporary", "adventure", "fiction", "non-fiction", "humor", "comedy",
    "paranormal-romance", "sci-fi-fantasy", "teen-fiction", "young-adult-fiction"
}

def extract_genre(shelves):
    if not isinstance(shelves, list):
        return "unknown"
    for shelf in shelves:
        if shelf["name"] in GENRE_SHELVES:
            return shelf["name"]
    return "unknown"

In [19]:
authored_df["genre"] = authored_df["popular_shelves"].apply(extract_genre)
authored_df = authored_df.drop(columns=["popular_shelves"])

print(authored_df["genre"].value_counts())
print(f"\nUnknown: {(authored_df['genre'] == 'unknown').sum()}")

genre
fantasy                675822
romance                270740
contemporary           260686
fiction                219360
dystopia               164279
paranormal             150830
mystery                 63895
science-fiction         51387
sci-fi                  48710
horror                  39102
historical-fiction      19512
humor                   10208
adventure                8589
thriller                 4145
non-fiction              3968
unknown                  3446
young-adult-fiction      2556
paranormal-romance        925
urban-fantasy             887
teen-fiction              411
comedy                    342
sci-fi-fantasy            200
Name: count, dtype: int64

Unknown: 3446


In [20]:
# removing unrated interactions
clean_df = authored_df[authored_df["rating"] > 0].copy()

In [21]:
print("After removing unrated:")
print("Users:", clean_df["user_id"].nunique())
print("Books:", clean_df["book_id"].nunique())

After removing unrated:
Users: 24867
Books: 48156


In [22]:
# filtering books with almost no ratings
filtered_df = clean_df.groupby("book_id").filter(lambda x: len(x) >= 5)

In [23]:
# filtering users who rated fewer than 5 books
filtered_df = filtered_df.groupby("user_id").filter(lambda x: len(x) >= 5) 

# filtering books without cover images
filtered_df = filtered_df[
    filtered_df["image_url"].notna() &
    (filtered_df["image_url"] != "") &
    (~filtered_df["image_url"].str.contains("nophoto", na = False))]

In [24]:
print("\nFinal dataset:")
print("Shape:", filtered_df.shape)
print("Users:", filtered_df["user_id"].nunique())
print("Books:", filtered_df["book_id"].nunique())
print("\nRatings per user:")
print(filtered_df.groupby("user_id").size().describe())


Final dataset:
Shape: (678435, 13)
Users: 18255
Books: 11288

Ratings per user:
count    18255.000000
mean        37.164339
std         53.267814
min          1.000000
25%          9.000000
50%         19.000000
75%         44.000000
max       1115.000000
dtype: float64


## Exploratory Data Analysis

Before building models, the key properties of the dataset are examined:
- **Sparsity** — how much of the user-item matrix is filled
- **Rating distribution** — whether ratings are skewed
- **Interaction matrix** — the pivot table used as input to the SVD model

In [25]:
n_users = filtered_df["user_id"].nunique()
n_books = filtered_df["book_id"].nunique()
n_ratings = len(filtered_df)
sparsity = 1 - (n_ratings / (n_users * n_books))

print(f"Users:    {n_users:,}")
print(f"Books:    {n_books:,}")
print(f"Ratings:  {n_ratings:,}")
print(f"Sparsity: {sparsity:.4f}")
print("\nNote: High sparsity is expected and normal for recommender systems.")
print("SVD handles sparse matrices well via latent factor decomposition.")

Users:    18,255
Books:    11,288
Ratings:  678,435
Sparsity: 0.9967

Note: High sparsity is expected and normal for recommender systems.
SVD handles sparse matrices well via latent factor decomposition.


In [26]:
#checking ratings are between 1-5
print("Rating distribution:")
print(filtered_df["rating"].value_counts().sort_index())

Rating distribution:
rating
1     16037
2     44489
3    150099
4    240905
5    226905
Name: count, dtype: int64


In [27]:
#building user-item interaction matrix 
interaction_matrix = filtered_df.pivot_table(
    index = "user_id",
    columns = "book_id",
    values = "rating"
)

In [28]:
print(f"Interaction matrix shape: {interaction_matrix.shape}")
interaction_matrix.head()

Interaction matrix shape: (18255, 11288)


book_id,2811,2931,3304,3402,3467,4325,5452,5454,8137,8948,...,34929050,35087274,35154365,35168764,35451387,35498621,35504431,35521513,35527721,36381037
user_id,,,,,,,,,,,,,,,,,,,,,
0008da70a705b4006dbd5aae83a47cc0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0021909de18ab01ec27517ea8dc0aa93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0021e047a599f9827d75628db22097b6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00254cd48d3d8a99ca9f0ed44fa69d5f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0028b5eebf06b43b321671ea39e7ca3c,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Popularity-based recommender (cold start)

New users have no rating history, so collaborative filtering cannot be applied. Instead, they receive recommendations based on overall book popularity.

A **Bayesian average** is used rather than a simple average rating. This penalises books with very few ratings (a book with 3 ratings averaging 5.0 should not outrank a book with 10,000 ratings averaging 4.5).

The Bayesian average formula is:

$$score = \frac{n \cdot \bar{x} + C \cdot m}{n + C}$$

Where:
- $n$ = number of ratings for the book
- $\bar{x}$ = average rating for the book  
- $C$ = mean number of ratings across all books
- $m$ = global mean rating across all books

In [29]:
#count ratings per book

num_rating_df = filtered_df.groupby("book_id")["rating"].count().reset_index()
num_rating_df.rename(columns= {"rating":"num_ratings"}, inplace = True)
num_rating_df

,book_id,num_ratings
0,2811,43
1,2931,4
2,3304,91
3,3402,2
4,3467,299
...,...,...
11283,35498621,6
11284,35504431,119
11285,35521513,6
11286,35527721,10


In [30]:
#finding the average rating for each book

avg_rating_df = filtered_df.groupby("book_id")["rating"].mean().reset_index()
avg_rating_df.rename(columns= {"rating": "avg_rating"}, inplace = True)
avg_rating_df

,book_id,avg_rating
0,2811,3.767442
1,2931,3.750000
2,3304,3.780220
3,3402,3.500000
4,3467,3.595318
...,...,...
11283,35498621,4.500000
11284,35504431,4.226891
11285,35521513,4.333333
11286,35527721,4.100000


In [31]:
#merging the two tables to create a new popular books dataframe

book_info = filtered_df[["book_id", "title", "author","genre", "description","image_url"]].drop_duplicates("book_id")
popular_df = num_rating_df.merge(avg_rating_df, on = "book_id").merge(book_info, on="book_id")

In [32]:
# measuring the Bayesian average score -- this means books with a higher volume of ratings are prioritised 
# over smaller volume of ratings but high ratings

C = popular_df["num_ratings"].mean() # average number of ratings across all books
m = popular_df["avg_rating"].mean() # global mean rating

popular_df["score"] = ((popular_df["num_ratings"] * popular_df["avg_rating"]) + (C*m)) / (popular_df["num_ratings"] + C)

In [33]:
# filter to books with a minimum number of ratings then rank

MIN_RATINGS = 50

popular_df = popular_df[popular_df["num_ratings"] >= MIN_RATINGS]
popular_df = popular_df.sort_values("score", ascending = False).reset_index(drop=True)
popular_df

,book_id,num_ratings,avg_rating,title,author,genre,description,image_url,score
0,32075671,553,4.634720,The Hate U Give,Angie Thomas,contemporary,Sixteen-year-old Starr Carter moves between tw...,https://images.gr-assets.com/books/1476284759m...,4.551767
1,6131164,859,4.563446,"Clockwork Princess (The Infernal Devices, #3)",Cassandra Clare,fantasy,"Danger and betrayal, secrets and enchantment i...",https://images.gr-assets.com/books/1460477760m...,4.512772
2,17340050,673,4.576523,"Losing Hope (Hopeless, #2)",Colleen Hoover,romance,In the follow-up to Colleen Hoover's #1 New Yo...,https://images.gr-assets.com/books/1368348507m...,4.511920
3,18006496,734,4.550409,"Queen of Shadows (Throne of Glass, #4)",Sarah J. Maas,fantasy,The queen has returned.\nEveryone Celaena Sard...,https://images.gr-assets.com/books/1441230104m...,4.492745
4,11387515,1149,4.512620,Wonder (Wonder #1),R.J. Palacio,fiction,I won't describe what I look like. Whatever yo...,https://images.gr-assets.com/books/1309285027m...,4.476626
...,...,...,...,...,...,...,...,...,...
1899,6449916,54,2.759259,Girl in the Arena,Lise Haines,dystopia,It's a fight to the death--on live TV--when a ...,https://images.gr-assets.com/books/1330973721m...,3.301413
1900,22188,360,3.208333,"Gossip Girl (Gossip Girl, #1)",Cecily von Ziegesar,fiction,"Welcome to New York City's Upper East Side, wh...",https://images.gr-assets.com/books/1398814332m...,3.291338
1901,8517207,129,3.054264,"Bumped (Bumped, #1)",Megan McCafferty,dystopia,When a virus makes everyone over the age of ei...,https://images.gr-assets.com/books/1288711015m...,3.287632
1902,6638377,91,2.956044,Nightlight: A Parody,The Harvard Lampoon,humor,About three things I was absolutely certain.\n...,https://images.gr-assets.com/books/1320519788m...,3.287169


In [34]:
#deduplicate by title (some books appear under multiple editions)

popular_df_no_duplicates = (popular_df
    .sort_values("num_ratings", ascending=False)
    .drop_duplicates(subset="title", keep="first")
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

In [36]:
print(f"Popular books available for cold-start: {len(popular_df_no_duplicates)}")
print()
print(popular_df_no_duplicates.head(10)[["title", "author", "num_ratings", "avg_rating", "score"]])

Popular books available for cold-start: 1778

                                               title                  author  \
0                                    The Hate U Give            Angie Thomas   
1      Clockwork Princess (The Infernal Devices, #3)         Cassandra Clare   
2                         Losing Hope (Hopeless, #2)          Colleen Hoover   
3             Queen of Shadows (Throne of Glass, #4)           Sarah J. Maas   
4                                 Wonder (Wonder #1)            R.J. Palacio   
5  The Hunger Games Trilogy Boxset (The Hunger Ga...         Suzanne Collins   
6                            Hopeless (Hopeless, #1)          Colleen Hoover   
7                 Crooked Kingdom (Six of Crows, #2)           Leigh Bardugo   
8            The Hunger Games (The Hunger Games, #1)         Suzanne Collins   
9                               Deity (Covenant, #3)  Jennifer L. Armentrout   

   num_ratings  avg_rating     score  
0          553    4.634720  4.5517

## Matrix Factorisation using SVD (personalised recommendations based on users with existing ratings)

Once a user has rated 5 or more books, the system switches to **SVD-based collaborative filtering** using the [Surprise](https://surpriselib.com/) library.

SVD (Singular Value Decomposition) is a matrix factorisation technique that decomposes the user-item rating matrix into latent factors representing hidden preferences. It can then predict how a user would rate books they haven't seen yet.

### Model Parameters (tuned via cross-validation)
| Parameter | Value | Description |
|---|---|---|
| `n_factors` | 50 | Number of latent factors |
| `n_epochs` | 30 | Number of SGD iterations |
| `lr_all` | 0.005 | Learning rate |
| `reg_all` | 0.02 | Regularisation term |

In [37]:
# define a Reader object to specify the rating scale
reader = Reader(rating_scale = (1,5))

# load into Surprise dataset
data = Dataset.load_from_df(
    filtered_df[["user_id", "book_id", "rating"]],
    reader
)

In [38]:
# splitting the data into train and test sets
trainset, testset = surprise_train_test_split(data, test_size=0.2)

In [39]:
# instantiate thej SVD model
svd = SVD()

# train the model on the training set
svd.fit(trainset)

In [40]:
# predict ratings for the test set
predictions = svd.test(testset)

# compute and print RMSE (root mean squared error) and MAE (mean absolute error)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.8428
MAE:  0.6573


predictions are off by less than one star on average

In [41]:
# cross-validate tuned SVD to confirm hyperparameter choices
svd_tuned = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02)
cross_validate(svd_tuned, data, measures=["RMSE", "MAE"], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8423  0.8402  0.8397  0.8435  0.8421  0.8416  0.0014  
MAE (testset)     0.6513  0.6502  0.6495  0.6533  0.6513  0.6511  0.0013  
Fit time          13.43   10.22   13.70   12.30   11.03   12.14   1.35    
Test time         1.20    0.89    2.70    1.82    1.92    1.71    0.63    


{'test_rmse': array([0.84234025, 0.84018548, 0.83968473, 0.84353394, 0.84214385]),
 'test_mae': array([0.65133684, 0.65020737, 0.64951836, 0.65329345, 0.65125153]),
 'fit_time': (13.433978796005249,
  10.219054937362671,
  13.700216054916382,
  12.30156683921814,
  11.030570030212402),
 'test_time': (1.2015109062194824,
  0.8910918235778809,
  2.7006890773773193,
  1.8166368007659912,
  1.923367977142334)}

In [42]:
def get_svd_recommendations(user_id, n=10):
    """
    Generate top-n SVD recommendations for a Goodreads user.
    Predicts ratings for all unrated books and returns the highest scored.
    """
    
    # get all books this user hasn't rated
    rated_books = filtered_df[filtered_df["user_id"] == user_id]["book_id"].values
    all_books = filtered_df["book_id"].unique()
    unrated_books = [b for b in all_books if b not in rated_books]

    # predict ratings for all unrated books
    predictions_list = [svd.predict(user_id, book_id) for book_id in unrated_books]

    # sort by estimated rating descending
    predictions_list.sort(key=lambda x: x.est, reverse=True)
    top_n = predictions_list[:n]

    # build results dataframe
    book_ids = [p.iid for p in top_n]
    scores = [round(p.est, 3) for p in top_n]

    book_info = filtered_df[["book_id", "title", "author", "genre", "description", "image_url"]].drop_duplicates("book_id")
    results = pd.DataFrame({"book_id": book_ids, "predicted_rating": scores})
    results = results.merge(book_info, on="book_id", how="left")
    results = results.drop_duplicates(subset="title", keep="first")

    return results

In [43]:
svd_tuned.fit(trainset)

In [44]:
sample_user = filtered_df["user_id"].iloc[0]
print(f"Recommendations for user: {sample_user}\n")
print(get_svd_recommendations(sample_user))

Recommendations for user: 0226259f03e2d21ccf7e5752fb386f6b

    book_id  predicted_rating  \
0  32075671             5.000   
1   8100904             4.915   
2     40159             4.907   
3  10165761             4.855   
4  27840861             4.844   
5  17347389             4.842   
6  11387515             4.779   
7    693208             4.749   
8  10165727             4.731   
9  12708927             4.727   

                                             title               author  \
0                                  The Hate U Give         Angie Thomas   
1              Monsters of Men (Chaos Walking, #3)         Patrick Ness   
2      The King of Attolia (The Queen's Thief, #3)  Megan Whalen Turner   
3     Quintana of Charyn (Lumatere Chronicles, #3)     Melina Marchetta   
4               Crooked Kingdom (Six of Crows, #2)        Leigh Bardugo   
5          The Dream Thieves (The Raven Cycle, #2)    Maggie Stiefvater   
6                               Wonder (Wonder #1) 

## Recommendation pipeline

The two strategies are combined into a single `get_recommendations` function that selects the appropriate method based on how many books the user has rated.

- **< 5 ratings** → cold start (popularity-based)
- **≥ 5 ratings** → warm start (SVD collaborative filtering)

The threshold of 5 is chosen as the minimum for SVD to have enough signal to produce meaningful personalised results.

In [45]:
def get_recommendations(user_id, n=10):
    """
    Returns top-n book recommendations for a user.
    - Cold start (< 5 ratings): falls back to popularity-based recommendations.
    - Warm start (>= 5 ratings): uses SVD collaborative filtering.
    """
    user_ratings = filtered_df[filtered_df["user_id"] == user_id]

    if len(user_ratings) < 5:
        # Cold start — return top popular books the user hasn't rated
        rated_ids = set(user_ratings["book_id"].values)
        recs = (
            popular_df_no_duplicates[~popular_df_no_duplicates["book_id"].isin(rated_ids)]
            .head(n)[["book_id", "title", "author", "genre", "description", "image_url", "score"]]
            .rename(columns={"score": "predicted_rating"})
            .copy()
        )
        recs["method"] = "popularity"
    else:
        # Warm start — SVD
        recs = get_svd_recommendations(user_id, n)
        recs["method"] = "svd"

    recs["user_id"] = user_id
    return recs

In [46]:
# test cold start (user with < 5 ratings)
print("COLD START:")
print(get_recommendations("cold_user"))

# test warm start (user with >= 5 ratings)
warm_user = filtered_df.groupby("user_id").filter(lambda x: len(x) >= 5)["user_id"].iloc[0]
print("\nWARM START (SVD):")
print(get_recommendations(warm_user))

COLD START:
    book_id                                              title  \
0  32075671                                    The Hate U Give   
1   6131164      Clockwork Princess (The Infernal Devices, #3)   
2  17340050                         Losing Hope (Hopeless, #2)   
3  18006496             Queen of Shadows (Throne of Glass, #4)   
4  11387515                                 Wonder (Wonder #1)   
5   7938275  The Hunger Games Trilogy Boxset (The Hunger Ga...   
6  15717943                            Hopeless (Hopeless, #1)   
7  22299763                 Crooked Kingdom (Six of Crows, #2)   
8   2767052            The Hunger Games (The Hunger Games, #1)   
9   9761778                               Deity (Covenant, #3)   

                   author         genre  \
0            Angie Thomas  contemporary   
1         Cassandra Clare       fantasy   
2          Colleen Hoover       romance   
3           Sarah J. Maas       fantasy   
4            R.J. Palacio       fiction   
5  